In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

In [10]:
# load 3 datasets and concatenate them
df1 = pd.read_csv('../data/general/news.csv')
df1['label'] = df1['label'].map({'REAL': 0, 'FAKE': 1}) # map labels to 0 and 1
df2 = pd.read_csv('../data/general/True.csv')
df2['label'] = 0 # all entries in this dataset are real news, so label is 0
df3 = pd.read_csv('../data/general/Fake.csv')
df3['label'] = 1 # all entries in this dataset are fake news, so label is 1
df = pd.concat([df1, df2, df3], ignore_index=True)
df.sample(n=5)

,index,title,text,label,subject,date
21512,NaN,Kremlin says prospects for Putin-Trump meeting...,"DANANG, Vietnam (Reuters) - The prospects for ...",0,worldnews,"November 10, 2017"
26825,NaN,Merkel optimistic EU dispute over refugee dist...,BERLIN (Reuters) - German Chancellor Angela Me...,0,worldnews,"September 10, 2017"
4839,7813.0,Email Reveals What Progressive Think Tank Gain...,"When a prominent, progressive establishment th...",1,NaN,NaN
36703,NaN,Fox Host Claims Obama Used A ‘Raw Onion’ To F...,Anyone who watched Barack Obama break into tea...,1,News,"January 5, 2016"
22946,NaN,French Catalans offer Puigdemont luxury safe-h...,PARIS (Reuters) - French Catalans have a villa...,0,worldnews,"October 24, 2017"


In [11]:
# drop columns that are not needed for classification
df = df.drop(columns=['index', 'title', 'subject', 'date'])

# remove entries where the text length is <200, as they are likely to be noise
df['text_length'] = df['text'].apply(len)
df = df[df['text_length'] >= 200].drop(columns=['text_length'])

In [12]:
# since we are partially using ISOT dataset, we need to perform some minor preprocessing to remove 
# the source of the news from the text, as it can be a strong signal for classification and can lead to overfitting. 
# We will remove any URLs and email addresses from the text.

def clean_text(text):
    # 1. Remove Publisher tags (ISOT specific)
    text = re.sub(r'^.*? - ', '', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # 3. Standard cleaning
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s!\?]', '', text) # Keep text, spaces, !, and ?
    return text

df['text'] = df['text'].apply(clean_text)

In [13]:
# split data into train and test sets
X, y = df['text'], df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
# preprocessing function to be used in the pipeline, which will clean the text using the same function as above
preprocess = FunctionTransformer(lambda x: pd.Series(x).apply(clean_text))

# create a pipeline with TfidfVectorizer and SGDClassifier, with hyperparameter weights already tuned previously using GridSearchCV
pipeline = make_pipeline(
    preprocess,
    TfidfVectorizer(max_features=10000, ngram_range=(1,2), stop_words='english'),
    SGDClassifier(loss='hinge', penalty=None, alpha=1e-4, random_state=42, learning_rate='pa1', eta0=1.0)
)

In [15]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.969859955348082
F1 Score: 0.9704683305160585
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      4842
           1       0.97      0.97      0.97      5012

    accuracy                           0.97      9854
   macro avg       0.97      0.97      0.97      9854
weighted avg       0.97      0.97      0.97      9854

Confusion Matrix:
 [[4677  165]
 [ 132 4880]]


In [18]:
# The example article is from the BBC, and is a real news article. The model should ideally predict it as 0 (real news).
# Link: https://www.bbc.com/news/articles/c99jxryd9xko
example_article = """
US Secretary of State Marco Rubio has spoken of a defining moment and a "new era" as he travels to Europe to give a major speech at the Munich Security Conference.

Rubio will lead the US team at the first major global event since President Donald Trump threatened Denmark's sovereignty with a pledge to annex Greenland.

French President Emmanuel Macron has insisted Europe must prepare for independence from the US, while Nato Secretary General Mark Rutte stressed that transatlantic bonds were as close and important as ever.

The war in Ukraine, tensions with China and a potential nuclear deal between Iran and the US are also on the agenda at the annual security conference, which began on Friday.

"The world is changing very fast right in front of us," Rubio told reporters, when asked if his message to Europeans would be more conciliatory than a year ago.

"We live in a new era in geopolitics, and it's going to require all of us to sort of re-examine what that looks like and what our role is going to be."

At last year's conference, US Vice-President JD Vance attacked Europe, including the UK, for policies on free speech and immigration. His comments triggered a year of unprecedented transatlantic tension.

Opening this year's conference, German Chancellor Friedrich Merz appealed directly to the US, saying "let's repair and revive transatlantic trust together".

He also revealed that "confidential talks" were ongoing with Macron on creating a joint European nuclear deterrent.

France and the UK are the only two nuclear powers in Europe - but Germany and many other European nations have traditionally relied on the US nuclear umbrella within the Nato alliance for deterrence.

Some 50 world leaders are set to attend this year's event, where European defence and the future of the transatlantic relationship will be discussed at a time when US commitments to Nato have been called into question.

Tensions have been heightened in recent months as Trump has repeatedly said that Greenland is vital to US national security, stating without evidence that it was "covered with Russian and Chinese ships all over the place".

Danish Prime Minister Mette Frederiksen said on Friday she planned to meet Rubio to discuss the US threats to seize Greenland - Denmark's semi-autonomous territory from its Nato ally.

The US threats have been viewed by many European leaders as a watershed moment that has eroded trust with its biggest ally.

Ahead of the conference, eight former US ambassadors to Nato and eight former American supreme commanders in Europe issued an open letter calling for Washington to maintain its support for the Western defensive alliance.

"Far from being a charity", they said Nato was a "force-multiplier" that allowed the US to assert its power and influence "in ways that would be impossible - or prohibitively expensive - to achieve on its own".

The transatlantic relationship has come under increasing strain following the Republican president's introduction of tariffs and a suggestion in the US national security strategy that European nations may not remain "reliable allies" in the long term.

Dutch Foreign Minister David van Weel told the BBC on Friday that he recognised "the world has changed" and that he hoped the "transatlantic bond" remains "solid".

Rubio is expected to avoid taking Vance's abrasive approach but, when asked if he was planning to be more conciliatory, he said that Europeans "want to know where we're going, where we'd like to go, where we'd like to go with them".

Macron will also address the conference on Friday, having told the World Economic Forum in Davos last month that now was "not a time for new imperialism or new colonialism".

After a week of turbulent domestic politics, UK Prime Minister Sir Keir Starmer will also travel to Munich, where he is expected to hold meetings both Merz and Macron, before addressing the conference on Saturday.

Conference chairman Wolfgang Ischinger, in a report ahead of the event, said: "For generations, US allies were not just able to rely on American power but on a broadly shared understanding of the principles underpinning the international order.

"Today, this appears far less certain, raising difficult questions about the future shape of transatlantic and international co-operation."

The former German diplomat said the White House's foreign policy "is already changing the world, and it has triggered dynamics whose full consequences are only beginning to emerge".

Upon arrival in Munich, Ukraine's President Volodymyr Zelensky said the conference would bring "new steps toward our shared security - that of Ukraine and Europe".

On Friday, Russia and Ukraine announced that a new round of peace talks also involving the US would take place in Geneva on 17-18 February, aimed at bringing Russia's full-scale, four-year invasion to an end.

The three countries recently held talks in Abu Dhabi, the capital of the United Arab Emirates (UAE), with no apparent breakthrough - though Ukraine and Russia carried out a rare prisoner of war exchange shortly after the meeting.

Rubio met Chinese Foreign Minister Wang Yi at the sidelines of the conference on Friday, as Washington and Beijing seek to ease tensions over a number of issues including trade and tariffs, as well as Taiwan.

Last week, China's President Xi Jinping called Taiwan "the most important issue" at stake between the US and China during a phone call Trump.

Xi also told the US president to be "prudent" when supplying weapons to the self-governing island, which Beijing views as a breakaway state it has not ruled out taking by force and which has long been supplied militarily by the US.

Iran's nuclear programme - which Tehran maintains is peaceful - is also expected to draw focus in Munich.

Trump has threatened military action against Iran if it does not agree a new deal to prevent it from developing nuclear weapons.
"""

In [17]:
pipeline.predict([example_article])

array([0])

Since the model predicted that the article was real, then it matches what we know about reality.

What if we try it on a fake news article?

In [20]:
# The example article is from The Gateway Pundit, and is a fake news article. The model should ideally predict it as 1 (fake news).
# I am not giving you the link to this garbage...
example_article_2 = """
Obama-appointed U.S. District Judge Paula Xinis has ruled that Immigration and Customs Enforcement (ICE) cannot re-detain Kilmar Abrego-Garcia, an illegal alien from El Salvador facing human smuggling charges.
Abrego Garcia, a Salvadoran national, has long been on the radar of U.S. law enforcement for his affiliations with the notorious MS-13 gang, also known as Mara Salvatrucha.

Breitbart News reports:

The government “made one empty threat after another to remove him to countries in Africa with no real chance of success,” U.S. District Judge Paula Xinis, in Maryland, wrote in her Tuesday order. “From this, the Court easily concludes that there is no ‘good reason to believe’ removal is likely in the reasonably foreseeable future.”

Abrego Garcia has an American wife and child and has lived in Maryland for years, but he immigrated to the U.S. illegally as a teenager. In 2019, an immigration judge ruled that he could not be deported to El Salvador because he faced danger there from a gang that had threatened his family. By mistake, he was deported there anyway last year.

Facing public pressure and a court order, President Donald Trump’s administration brought him back in June, but only after securing an indictment charging him with human smuggling in Tennessee. He has pleaded not guilty. Meanwhile, Trump officials have said he cannot stay in the U.S. In court filings, officials have said they intended to deport him to Uganda, Eswatini, Ghana, and Liberia.

MS-13 is infamous for its involvement in violent crimes, including murder, extortion, and human trafficking.

Garcia’s criminal history includes a 2018 conviction for gang participation in Prince George’s County Circuit Court in Maryland, where he was found guilty of actively supporting MS-13 activities.

In April, DHS released a detailed investigative report outlining a traffic stop that led authorities to believe Garcia was involved in transporting undocumented migrants across borders for profit.

Garcia was subsequently detained by Immigration and Customs Enforcement (ICE).

In December, Judge Paula Xinis ordered Garcia’s immediate release from immigration detention, citing procedural concerns and what she described as insufficient evidence to justify prolonged holding without trial.

DHS Secretary Kristi Noem publicly condemned the ruling, calling it a threat to national security and accusing the judge of prioritizing the rights of criminals over American safety.

Additionally, Garcia’s defense team successfully pushed for a gag order against top Trump administration officials, including Attorney General Pam Bondi and Secretary Noem. This was the third such complaint from his attorneys, who claimed public statements from government figures were prejudicing his right to a fair trial.

Just days after his release, Garcia wasted no time in leveraging social media.

In a TikTok video that went viral, he was seen outdoors in a residential area, wearing a baseball cap and a Nike hoodie, passionately singing a Spanish-language Christian hymn.

The lyrics, which reference biblical miracles like parting the Red Sea and overcoming trials through faith, include lines such as “When you have to cross the Red Sea, call on this man with faith, only he opens the sea” and “You will pass to the other side, and there you will sing the hymn of victory.”

Garcia also spoke out publicly in Baltimore, Maryland, addressing a crowd and portraying himself as a victim of overzealous enforcement.

His actions have drawn sharp contrasts with the gag order binding DHS, leading to accusations that the justice system is upside down and empowering alleged criminals while handcuffing law enforcement.
"""

In [21]:
pipeline.predict([example_article])

array([0])

Surprisingly, the model marks it as real! I wonder why that may be the case.

In [23]:
# 20 largest factors that can predict whether an article is fake or real news, according to the model. We will use the coefficients of the SGDClassifier to find the most important features.
feature_names = pipeline.named_steps['tfidfvectorizer'].get_feature_names_out()
coefficients = pipeline.named_steps['sgdclassifier'].coef_[0]
feature_importance = pd.DataFrame({'feature': feature_names, 'coefficient': coefficients})
feature_importance['abs_coefficient'] = feature_importance['coefficient'].abs()
top_features = feature_importance.sort_values(by='abs_coefficient', ascending=False).head(20)
print(top_features[['feature', 'coefficient']])

               feature  coefficient
4136             image    13.728022
6646  president donald   -11.468980
2511             doesn    10.714105
7636              said   -10.304100
2367              didn     9.135440
5961           october     9.005516
9045      told reuters    -8.756525
9285            trumps    -8.657807
416                 ap     8.633501
6657   president trump     8.459098
9775               wfb     8.437923
5023                ll     7.818502
7719           saidthe     7.807142
4485               isn     7.774692
9305           tuesday    -7.614313
2533               don     7.231390
9693              wasn     7.176788
2909               est    -7.055250
5908               nyp     6.907614
9514                ve     6.738964
